In [1]:
import cv2 as cv
import matplotlib.pyplot as plt

In [2]:
# Load the pre-trained neural network model from a TensorFlow file
net = cv.dnn.readNetFromTensorflow("graph_opt.pb") # weights

# Set the input dimensions for the neural network
inWidth = 368
inHeight = 368
# Set the confidence threshold for detecting body parts
thr = 0.2

# Define the dictionary of body parts with their corresponding indices
BODY_PARTS = { "Nose": 0, "Neck": 1, "RShoulder": 2, "RElbow": 3, "RWrist": 4,
               "LShoulder": 5, "LElbow": 6, "LWrist": 7, "RHip": 8, "RKnee": 9,
               "RAnkle": 10, "LHip": 11, "LKnee": 12, "LAnkle": 13, "REye": 14,
               "LEye": 15, "REar": 16, "LEar": 17, "Background": 18 }

# Define the list of pairs of body parts that form the skeleton connections
POSE_PAIRS = [ ["Neck", "RShoulder"], ["Neck", "LShoulder"], ["RShoulder", "RElbow"],
               ["RElbow", "RWrist"], ["LShoulder", "LElbow"], ["LElbow", "LWrist"],
               ["Neck", "RHip"], ["RHip", "RKnee"], ["RKnee", "RAnkle"], ["Neck", "LHip"],
               ["LHip", "LKnee"], ["LKnee", "LAnkle"], ["Neck", "Nose"], ["Nose", "REye"],
               ["REye", "REar"], ["Nose", "LEye"], ["LEye", "LEar"] ]


In [ ]:
# Capture video from the second connected camera (usually the first external camera)
cap = cv.VideoCapture(1)
# Set the frames per second to 10
cap.set(cv.CAP_PROP_FPS, 10)
# Set the width of the capture window to 800 pixels
cap.set(3, 800)
# Set the height of the capture window to 800 pixels
cap.set(4, 800)

# If the camera is not opened, try opening the default camera
if not cap.isOpened():
    cap = cv.VideoCapture(0)
# If the default camera is also not opened, raise an error
if not cap.isOpened():
    raise IOError("Cannot open webcam")

# Continuously capture frames from the webcam
while cv.waitKey(1) < 0:
    hasFrame, frame = cap.read()
    # If a frame is not captured, wait for a key press and break the loop
    if not hasFrame:
        cv.waitKey()
        break

    # Get the width and height of the captured frame
    frameWidth = frame.shape[1]
    frameHeight = frame.shape[0]
    # Prepare the frame for the neural network
    net.setInput(cv.dnn.blobFromImage(frame, 1.0, (inWidth, inHeight), (127.5, 127.5, 127.5), swapRB=True, crop=False))
    # Perform a forward pass through the network
    out = net.forward()
    # Keep only the first 19 channels (corresponding to body parts)
    out = out[:, :19, :, :]

    # Ensure the number of body parts matches the output shape
    assert(len(BODY_PARTS) == out.shape[1])

    # Initialize a list to store the detected points
    points = []

    # Loop through each body part
    for i in range(len(BODY_PARTS)):
        # Extract the heatmap for the current body part
        heatMap = out[0, i, :, :]

        # Find the position of the maximum confidence in the heatmap
        _, conf, _, point = cv.minMaxLoc(heatMap)
        x = (frameWidth * point[0]) / out.shape[3]
        y = (frameHeight * point[1]) / out.shape[2]

        # Append the point if the confidence is above the threshold, otherwise append None
        points.append((int(x), int(y)) if conf > thr else None)

    # Draw the skeleton
    for pair in POSE_PAIRS:
        partFrom = pair[0]
        partTo = pair[1]
        assert(partFrom in BODY_PARTS)
        assert(partTo in BODY_PARTS)

        idFrom = BODY_PARTS[partFrom]
        idTo = BODY_PARTS[partTo]

        # Draw lines and points for valid pairs
        if points[idFrom] and points[idTo]:
            cv.line(frame, points[idFrom], points[idTo], (0, 255, 0), 3)
            cv.ellipse(frame, points[idFrom], (3, 3), 0, 0, 360, (0, 0, 255), cv.FILLED)
            cv.ellipse(frame, points[idTo], (3, 3), 0, 0, 360, (0, 0, 255), cv.FILLED)

    # Calculate the time taken for the forward pass
    t, _ = net.getPerfProfile()
    freq = cv.getTickFrequency() / 1000
    # Display the time taken for the forward pass on the frame
    cv.putText(frame, '%.2fms' % (t / freq), (10, 20), cv.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0))

    # Show the frame with the skeleton
    cv.imshow('Pose estimation trial', frame)
